# Host Prophage Coverage And Activity

PropagAtE-style checks use its default activity thresholds: ratio >=2, Cohen's d >=0.70, mean prophage depth >=1x and breadth >=50% at >=1x. Scaffold ends are masked by 150 bp, and prophages must be >=1 kb. All QC-passed paired and orphan reads are mapped to the assigned unmasked host; residual host/prophage stages are not used. Secondary and supplementary alignments do not add depth.

MOSAIC's agreed mapping filters remain >=95% identity and >=85% aligned read length, rather than PropagAtE's default 97% identity. This is not an identical stock PropagAtE run. Activity calls describe coverage enrichment, not confirmed induction in a phage-enriched isolate library. Zero host background, short prophages and whole-contig viral candidates remain explicitly not assessed. This report never changes contig retention.

Method: [PropagAtE documentation](https://github.com/AnantharamanLab/PropagAtE).


In [ ]:
from pathlib import Path
import gzip
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from IPython.display import display, Image, Markdown

sns.set_style("ticks", {"axes.grid": True})
sns.set_palette("colorblind")
plt.rcParams.update({"axes.linewidth": 1.5, "axes.labelsize": 14, "axes.titlesize": 14,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "svg.fonttype": "none",
                     "font.family": "sans-serif", "font.sans-serif": ["Liberation Sans", "DejaVu Sans"]})
regions = pd.read_csv(snakemake.input.regions, sep="\t", dtype={"host": str, "viral_id": str, "parent_contig": str})
assignments = pd.read_csv(snakemake.input.assignments, sep="\t", dtype={"sample": str, "expected_host": str}).set_index("sample")["expected_host"].to_dict()
HOSTS = sorted(snakemake.params.hosts, key=str.casefold)
host_lengths = {}
for host, path in zip(snakemake.params.hosts, snakemake.input.host_fastas):
    with open(path) as handle:
        host_lengths[host] = {record.id: len(record) for record in SeqIO.parse(handle, "fasta")}
figure_dir = Path(snakemake.output.figures)
figure_dir.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([dict(min_ratio=snakemake.params.min_ratio,
                          min_cohen_d=snakemake.params.min_cohen_d,
                          min_mean_depth=snakemake.params.min_mean_depth,
                          min_breadth_percent=snakemake.params.min_breadth_percent,
                          scaffold_end_mask_bp=snakemake.params.mask_bp,
                          min_prophage_length_bp=snakemake.params.min_length_bp)]))


## Per-Base Coverage Checks

The background is the same host scaffold outside the union of all its embedded prophage intervals. Unreported zero-depth positions are restored. Masked scaffold ends are excluded from means, variances and breadth denominators; the assessed length is reported. Cohen's d uses the unweighted pooled variance used by PropagAtE, not a length-weighted pooled variance. No smoothing or binning affects the calls.

Active: all four activity thresholds pass. Ambiguous: ratio and effect pass, but mean depth or breadth does not. Dormant: no activity enrichment passes. These are coverage-based estimates; a dormant call is not proof of inactivity. No prophage coverage is not_present. No measurable host background is not_assessed_host_background.

The coverage-check table includes the existing CheckV quality category and estimated completeness (%), with 'not reported' for missing assessments. These describe sequence quality, not activity, and do not change the coverage-based calls. The saved TSV retains all region annotations and measurements.


In [ ]:
def prophage_activity_metrics(depth, start, end, embedded_intervals, evidence_scope, params):
    # Coordinates in the evidence table are 1-based and inclusive.
    start, end = min(start, end) - 1, max(start, end)
    valid = np.ones(len(depth), dtype=bool)
    if params.mask_bp:
        valid[:params.mask_bp] = False
        valid[-params.mask_bp:] = False
    background = valid.copy()
    for left, right in embedded_intervals:
        background[min(left, right) - 1:max(left, right)] = False
    if evidence_scope != "embedded_prophage":
        background[:] = False
    phage = depth[start:end][valid[start:end]]
    host = depth[background]
    phage_mean = float(phage.mean()) if phage.size else np.nan
    host_mean = float(host.mean()) if host.size else np.nan
    phage_sd = float(phage.std()) if phage.size else np.nan
    host_sd = float(host.std()) if host.size else np.nan
    breadth = 100 * np.count_nonzero(phage >= params.min_mean_depth) / phage.size if phage.size else np.nan
    ratio = phage_mean / host_mean if host_mean > 0 else np.nan
    pool = np.sqrt((phage_sd ** 2 + host_sd ** 2) / 2)
    effect = abs(phage_mean - host_mean) / pool if pool > 0 else (
        np.inf if np.isfinite(phage_mean) and np.isfinite(host_mean) and phage_mean != host_mean else 0.0)
    passes_ratio = bool(ratio >= params.min_ratio)
    passes_effect = bool(effect >= params.min_cohen_d)
    passes_depth = bool(phage_mean >= params.min_mean_depth)
    passes_breadth = bool(breadth >= params.min_breadth_percent)
    if evidence_scope != "embedded_prophage":
        status = "not_assessed_whole_contig"
    elif end - start < params.min_length_bp:
        status = "not_assessed_short_prophage"
    elif not phage.size:
        status = "not_assessed_after_end_mask"
    elif phage_mean == 0:
        status = "not_present"
    elif not host.size or not host_mean > 0:
        status = "not_assessed_host_background"
    elif passes_ratio and passes_effect:
        status = "active" if passes_depth and passes_breadth else "ambiguous"
    else:
        status = "dormant"
    return dict(prophage_mean_depth=phage_mean, prophage_median_depth=float(np.median(phage)) if phage.size else np.nan,
                prophage_sd_depth=phage_sd, prophage_breadth_percent=breadth,
                prophage_covered_bases=int(np.count_nonzero(phage > 0)),
                assessed_prophage_length_bp=int(phage.size), host_background_length_bp=int(host.size),
                host_mean_depth=host_mean, host_sd_depth=host_sd,
                host_breadth_percent=100 * np.count_nonzero(host >= 1) / host.size if host.size else np.nan,
                prophage_host_ratio=ratio, cohen_d=effect,
                passes_ratio=passes_ratio, passes_cohen_d=passes_effect,
                passes_mean_depth=passes_depth, passes_breadth=passes_breadth,
                activity=status, mapping_identity_percent=95, mapping_aligned_percent=85)


In [ ]:
rows, profiles = [], []
for sample, path in zip(snakemake.params.samples, snakemake.input.depth):
    host = assignments[sample]
    selected = regions[regions["host"].eq(host)]
    depths = {parent: np.zeros(host_lengths[host][parent], dtype=float)
              for parent in selected["parent_contig"].unique()}
    with gzip.open(path, "rt") as handle:
        has_coverage = bool(handle.read(1))
    if has_coverage:
        for chunk in pd.read_csv(path, sep="\t", names=["contig", "position", "depth"], dtype={"contig": str}, chunksize=100000):
            for parent, values in chunk[chunk["contig"].isin(depths)].groupby("contig"):
                depths[parent][values["position"].to_numpy(dtype=int)] = values["depth"].to_numpy(dtype=float)
    for region in selected.itertuples(index=False):
        depth = depths[region.parent_contig]
        embedded = selected[selected["parent_contig"].eq(region.parent_contig) & selected["evidence_scope"].eq("embedded_prophage")]
        intervals = list(zip(embedded["start"].astype(int), embedded["end"].astype(int)))
        metrics = prophage_activity_metrics(depth, int(region.start), int(region.end), intervals,
                                           region.evidence_scope, snakemake.params)
        rows.append(dict(sample=sample, **region._asdict(), **metrics))
        if region.evidence_scope == "embedded_prophage" and metrics["prophage_mean_depth"] >= snakemake.params.min_mean_depth:
            start, end = sorted([int(region.start), int(region.end)])
            left = max(0, start - 1 - snakemake.params.plot_flank_bp)
            right = min(len(depth), end + snakemake.params.plot_flank_bp)
            edges = np.unique(np.linspace(left, right, min(2001, right - left + 1), dtype=int))
            profiles.append(dict(sample=sample, viral_id=region.viral_id, host=host,
                                 parent_contig=region.parent_contig, host_contig_length_bp=len(depth),
                                 prophage_region_length_bp=end - start + 1,
                                 checkv_quality=getattr(region, "CheckV_checkv_quality", "not reported"),
                                 checkv_completeness=pd.to_numeric(getattr(region, "CheckV_completeness", np.nan), errors="coerce"),
                                 start=start, end=end, activity=metrics["activity"],
                                 ratio=metrics["prophage_host_ratio"], cohen_d=metrics["cohen_d"],
                                 host_depth=metrics["host_mean_depth"],
                                 plot_start=left + 1, plot_end=right,
                                 positions=(edges[:-1] + edges[1:]) / 2,
                                 depth=np.array([depth[a:b].mean() for a, b in zip(edges[:-1], edges[1:])])))
activity = pd.DataFrame(rows)
if activity.empty:
    activity = pd.DataFrame(columns=["sample"] + regions.columns.tolist() + list(prophage_activity_metrics(
        np.zeros(2000), 1, 1000, [(1, 1000)], "embedded_prophage", snakemake.params)))
activity.to_csv(snakemake.output.table, sep="\t", index=False)
coverage_checks = activity.reindex(columns=["sample", "host", "viral_id",
    "CheckV_checkv_quality", "CheckV_completeness", "assessed_prophage_length_bp",
    "prophage_mean_depth", "host_mean_depth", "prophage_host_ratio", "cohen_d",
    "prophage_breadth_percent", "activity"]).rename(columns={
        "CheckV_checkv_quality": "CheckV quality", "CheckV_completeness": "CheckV completeness"})
coverage_checks["CheckV quality"] = coverage_checks["CheckV quality"].fillna("not reported")
coverage_checks["CheckV completeness"] = pd.to_numeric(coverage_checks["CheckV completeness"], errors="coerce").map(
    lambda value: f"{value:.1f}%" if pd.notna(value) else "not reported")
with pd.option_context("display.max_columns", None):
    display(coverage_checks)
display(activity.groupby(["host", "activity"]).size().unstack(fill_value=0))


## Coverage Enrichment Across Samples

Each host has its own heatmap: only its embedded prophages and samples assigned to that host are compared. Row labels include the region's CheckV quality assessment in parentheses; missing assessments are shown as 'not reported'. CheckV quality describes the sequence, separately from coverage-based activity. Hosts without any assessable coverage have a short text entry instead of a heatmap, including the number of assigned samples. All available region measurements remain in the results table. Hosts and samples are ordered alphabetically.

Colour and numerical annotations are log2(prophage/host mean depth): 0 means equal depth, 1 means two-fold enrichment and 4 means 16-fold enrichment. Values are displayed to one decimal place. Every host uses the same colour scale, calculated from all displayed values without clipping to a fixed maximum. An asterisk marks active by the activity thresholds; a question mark marks ambiguous. A dash (–) means no prophage coverage; NA means activity cannot be assessed, for example because host background coverage is zero. These cells have no enrichment colour. Unmarked numerical values are assessed but do not pass the activity thresholds. The activity flag is not proof of induction.

Individual host figures and their index are saved in the activity figure directory. The existing overview PNG/SVG contains the same host-specific panels.

In [ ]:
# Separate assigned-host comparisons; omit hosts with no assessable embedded regions
heatmaps = []
for host in HOSTS:
    host_samples = sorted((sample for sample in snakemake.params.samples if assignments[sample] == host),
                          key=str.casefold)
    selected = activity[activity["host"].eq(host) & activity["evidence_scope"].eq("embedded_prophage")]
    identifiers = selected.loc[selected["prophage_mean_depth"].gt(0), "viral_id"].drop_duplicates().tolist()
    selected = selected[selected["viral_id"].isin(identifiers)]
    matrix = selected.pivot(index="viral_id", columns="sample", values="prophage_host_ratio").reindex(
        index=identifiers, columns=host_samples)
    classes = selected.pivot(index="viral_id", columns="sample", values="activity").reindex(
        index=matrix.index, columns=matrix.columns)
    matrix = matrix.where(np.isfinite(matrix) & matrix.gt(0) & classes.isin(["active", "ambiguous", "dormant"]))
    matrix = matrix.loc[matrix.notna().any(axis=1)]
    if matrix.empty:
        continue
    classes = classes.reindex(index=matrix.index)
    stars = classes.replace({"active": "*", "ambiguous": "?"}).where(classes.isin(["active", "ambiguous"]), "")
    annotations = np.log2(matrix).apply(lambda column: column.map(
        lambda value: f"{value:.1f}" if np.isfinite(value) else "NA"))
    annotations = (annotations + stars).where(matrix.notna(), "NA").mask(classes.eq("not_present"), "–")
    quality = selected.reindex(columns=["viral_id", "CheckV_checkv_quality"]).drop_duplicates("viral_id").set_index("viral_id")["CheckV_checkv_quality"]
    quality = quality.reindex(matrix.index).fillna("not reported")
    row_labels = [f"{identifier} (CheckV: {quality.loc[identifier]})" for identifier in matrix.index]
    heatmaps.append(dict(host=host, matrix=matrix, stars=stars, annotations=annotations, row_labels=row_labels))

# One shared, data-derived log2 scale preserves comparisons without saturating high ratios.
log_values = np.concatenate([np.log2(item["matrix"].to_numpy()).ravel() for item in heatmaps]) if heatmaps else np.array([])
log_values = log_values[np.isfinite(log_values)]
colour_min = min(-1, float(np.floor(log_values.min()))) if log_values.size else -1
colour_max = max(1, float(np.ceil(log_values.max()))) if log_values.size else 1

# Keep the existing overview filenames for reports that link to them.
# Allow space for region IDs and CheckV assessments without squeezing the heatmap.
panel_heights = [max(3, 0.35 * len(item["matrix"]) + 2) for item in heatmaps]
fig, axs = plt.subplots(max(1, len(heatmaps)), 1, squeeze=False,
    figsize=(max([8] + [0.3 * len(item["matrix"].columns) + 0.07 * max(map(len, item["row_labels"])) + 5 for item in heatmaps]),
             sum(panel_heights) if panel_heights else 3),
    gridspec_kw={"height_ratios": panel_heights or [1]})
for ax, item in zip(axs[:, 0], heatmaps):
    matrix = item["matrix"]
    sns.heatmap(np.log2(matrix), cmap="vlag", center=0, vmin=colour_min, vmax=colour_max,
                annot=item["annotations"], fmt="", annot_kws={"size": 8}, ax=ax, xticklabels=True, yticklabels=item["row_labels"],
                cbar_kws={"label": "log2(prophage / host mean depth)"})
    # Seaborn skips masked cells; label them without assigning an enrichment colour.
    for row, column in zip(*np.where(matrix.isna())):
        ax.text(column + 0.5, row + 0.5, item["annotations"].iat[row, column],
                ha="center", va="center", color="#555555", fontsize=8)
    ax.set_xlabel("Samples assigned to this host")
    ax.set_ylabel("Host prophage")
    ax.set_title(f"Host {item['host']} | * active; ? ambiguous\n– no coverage; NA not assessed")
if not heatmaps:
    axs[0, 0].text(0.5, 0.5, "No assessable embedded prophage coverage", ha="center", va="center",
                   transform=axs[0, 0].transAxes)
    axs[0, 0].set_axis_off()
fig.tight_layout()
fig.savefig(snakemake.output.png, dpi=300, bbox_inches="tight")
fig.savefig(snakemake.output.svg, bbox_inches="tight")
plt.close(fig)

# Display and save only hosts with assessable heatmap cells.
heatmap_dir = figure_dir / "heatmaps"
heatmap_dir.mkdir(parents=True, exist_ok=True)
heatmap_index = []
for item in heatmaps:
    matrix = item["matrix"]
    fig, ax = plt.subplots(figsize=(max(8, 0.3 * len(matrix.columns) + 0.07 * max(map(len, item["row_labels"])) + 5),
                                   max(3, 0.35 * len(matrix) + 2)))
    sns.heatmap(np.log2(matrix), cmap="vlag", center=0, vmin=colour_min, vmax=colour_max,
                annot=item["annotations"], fmt="", annot_kws={"size": 8}, ax=ax, xticklabels=True, yticklabels=item["row_labels"],
                cbar_kws={"label": "log2(prophage / host mean depth)"})
    for row, column in zip(*np.where(matrix.isna())):
        ax.text(column + 0.5, row + 0.5, item["annotations"].iat[row, column],
                ha="center", va="center", color="#555555", fontsize=8)
    ax.set_xlabel("Samples assigned to this host")
    ax.set_ylabel("Host prophage")
    ax.set_title(f"Host {item['host']} | * active; ? ambiguous\n– no coverage; NA not assessed")
    fig.tight_layout()
    png = heatmap_dir / (item["host"] + ".png")
    svg = png.with_suffix(".svg")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    heatmap_index.append(dict(host=item["host"], samples=len(matrix.columns), regions=len(matrix),
                              png=str(png), svg=str(svg)))
pd.DataFrame(heatmap_index, columns=["host", "samples", "regions", "png", "svg"]).to_csv(
    heatmap_dir / "heatmaps.tsv", sep="\t", index=False)

# Keep every host visible in the notebook, using text when no heatmap can be assessed.
heatmap_paths = {item["host"]: item["png"] for item in heatmap_index}
for host in HOSTS:
    if host in heatmap_paths:
        display(Image(filename=heatmap_paths[host]))
    else:
        n_samples = sum(assignments[sample] == host for sample in snakemake.params.samples)
        display(Markdown(f"**Host {host}:** included in the analysis (assigned samples: {n_samples}), "
                         "but no assessable embedded prophage coverage was available. No heatmap is shown."))

## Prophage Coverage Profiles

Profiles are grouped by host, then prophage region, so the same region can be compared across samples together. Hosts are alphabetical; regions follow parent-contig and genomic-position order; samples within each region follow the full sample-ID order. Host and prophage-region headings make the groups explicit. Titles give the full host-contig length, predicted prophage-region length, CheckV quality and completeness, fold change and Cohen's d. Missing CheckV assessments are 'not reported'. Embedded prophages with mean depth at least the configured minimum are shown, with 50,000 bp on each side by default (`host_prophage_activity_plot_flank_bp`), clipped at contig boundaries.

Shading shows the predicted prophage interval: orange for an active call, pink for an ambiguous call, and grey for dormant or not-assessed calls. The legend and title retain the exact activity status. The dashed line is the mean host background from the whole parent contig, excluding all embedded prophages and the 150-bp end masks. Fold change is mean prophage depth divided by that background mean, including uncovered positions as zero. Neither the local plot window nor display binning changes activity statistics. A zero background is not assessed, not infinite enrichment. Every region, including those without a displayed profile, remains in the results table.

In [ ]:
profile_index = []
profiles = sorted(profiles, key=lambda profile: (profile["host"].casefold(),
    profile["parent_contig"], profile["start"], profile["end"], profile["viral_id"], profile["sample"].casefold()))
current_host, current_region = None, None
for profile in profiles:
    if profile["host"] != current_host:
        display(Markdown(f"### Host {profile['host']}"))
        current_host, current_region = profile["host"], None
    if profile["viral_id"] != current_region:
        display(Markdown(f"#### Prophage region {profile['viral_id']}"))
        current_region = profile["viral_id"]
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(profile["positions"], profile["depth"], color="#0173b2", linewidth=1)
    region_color={"active": "#de8f05", "ambiguous": "#cc79a7"}.get(profile["activity"], "#8a8a8a")
    ax.axvspan(profile["start"] - 1, profile["end"], color=region_color, alpha=0.2,
               label=f"Predicted prophage ({profile['activity']})")
    ax.axhline(profile["host_depth"], color="grey", linestyle="--", label="Whole-contig background mean")
    ax.set_xlim(profile["plot_start"] - 1, profile["plot_end"])
    ratio_text = f"{profile['ratio']:.2f}" if np.isfinite(profile["ratio"]) else "not assessed"
    effect_text = f"{profile['cohen_d']:.2f}" if not np.isnan(profile["cohen_d"]) else "not assessed"
    checkv_quality = profile.get("checkv_quality", "not reported")
    checkv_quality = "not reported" if pd.isna(checkv_quality) else checkv_quality
    checkv_completeness = pd.to_numeric(profile.get("checkv_completeness", np.nan), errors="coerce")
    completeness_text = f"{checkv_completeness:.1f}%" if pd.notna(checkv_completeness) else "not reported"
    ax.set_title(f"Host {profile['host']} | {profile['sample']} | {profile['viral_id']}\n"
                 f"Host contig length: {profile['host_contig_length_bp']:,} bp | "
                 f"Prophage region length: {profile['prophage_region_length_bp']:,} bp\n"
                 f"CheckV: {checkv_quality} | completeness: {completeness_text}\n"
                 f"{profile['activity']}; prophage/host fold change: {ratio_text}; Cohen's d: {effect_text}", fontsize=10)
    ax.set_xlabel("Position on host contig (bp)")
    ax.set_ylabel("Read depth")
    ax.legend(frameon=False)
    png = figure_dir / (profile["sample"] + "__" + profile["viral_id"] + ".png")
    svg = png.with_suffix(".svg")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    profile_index.append(dict(host=profile["host"], sample=profile["sample"], viral_id=profile["viral_id"],
                              parent_contig=profile["parent_contig"],
                              host_contig_length_bp=profile["host_contig_length_bp"],
                              prophage_region_length_bp=profile["prophage_region_length_bp"],
                              checkv_quality=checkv_quality, checkv_completeness=checkv_completeness,
                              cohen_d=profile["cohen_d"],
                              plot_start=profile["plot_start"], plot_end=profile["plot_end"], png=str(png), svg=str(svg)))
    display(Image(filename=str(png)))
pd.DataFrame(profile_index, columns=["host", "sample", "viral_id", "parent_contig", "host_contig_length_bp",
    "prophage_region_length_bp", "checkv_quality", "checkv_completeness", "cohen_d", "plot_start", "plot_end", "png", "svg"]).to_csv(
    figure_dir / "coverage_profiles.tsv", sep="\t", index=False)